# M2 Initial Implementation

## Discovery Question 1:
*What 'staple' cards tend to be present in decks of the various color combinations?*

This question is essentialy asking what cards tend to be in the same deck. This is the same as the task of finding out what items tend to be in the same shopping cart which means that we can solve it the same way. I will use Apriori to find frequent item sets, and association rules to identify the cards that frequently appear together aka the 'staples'.

## Imports

In [9]:
import json
import glob
import os
import pandas

## Step 1: Load Supporting data from Previous Reorganization efforts

### Step 1.1: Load the AllCardsUUID.json and AllCardsName.json

In [10]:
SLIM_DECKS_PATH = '../Data/AllDeckFiles(slim)/'
ALL_CARDS_NAME_PATH = '../Data/AllCardsName.json'
ENCODING = 'utf-8'

with open(ALL_CARDS_NAME_PATH, 'r', encoding=ENCODING) as f:
    card_data = json.load(f)

card_lookup = {
    name: {
        'colorIdentity': info.get('colorIdentity', []),
        'types':         info.get('types', []),
        'manaValue':     info.get('manaValue', None),
    }
    for name, info in card_data.items()
}

print(f'Loaded lookup for {len(card_lookup):,} unique card UUIDs.')

Loaded lookup for 26,345 unique card UUIDs.


### Step 1.2: A helper function for more readable data
A key part of the question is associating 'staple' cards with certain color combinations. We can use these to make the color combinations more readable for the MTG format.

In [11]:
def color_label(colors):
    mapping = {
        frozenset():                  'Colorless',
        frozenset(['W']):             'Mono-White',
        frozenset(['U']):             'Mono-Blue',
        frozenset(['B']):             'Mono-Black',
        frozenset(['R']):             'Mono-Red',
        frozenset(['G']):             'Mono-Green',
        # Guild pairs
        frozenset(['W','U']):         'Azorius (WU)',
        frozenset(['W','B']):         'Orzhov (WB)',
        frozenset(['U','B']):         'Dimir (UB)',
        frozenset(['U','R']):         'Izzet (UR)',
        frozenset(['B','R']):         'Rakdos (BR)',
        frozenset(['B','G']):         'Golgari (BG)',
        frozenset(['R','G']):         'Gruul (RG)',
        frozenset(['R','W']):         'Boros (RW)',
        frozenset(['G','W']):         'Selesnya (GW)',
        frozenset(['G','U']):         'Simic (GU)',
        # Shards (arc/allied three-color)
        frozenset(['W','U','B']):     'Esper (WUB)',
        frozenset(['U','B','R']):     'Grixis (UBR)',
        frozenset(['B','R','G']):     'Jund (BRG)',
        frozenset(['R','G','W']):     'Naya (RGW)',
        frozenset(['G','W','U']):     'Bant (GWU)',
        # Wedges (enemy three-color)
        frozenset(['W','B','R']):     'Mardu (WBR)',
        frozenset(['U','R','G']):     'Temur (URG)',
        frozenset(['B','G','W']):     'Abzan (BGW)',
        frozenset(['R','W','U']):     'Jeskai (RWU)',
        frozenset(['G','U','B']):     'Sultai (GUB)',
        # Four-color (Nephilim)
        frozenset(['W','U','B','R']): 'Artifice (WUBR)',
        frozenset(['U','B','R','G']): 'Chaos (UBRG)',
        frozenset(['B','R','G','W']): 'Aggression (BRGW)',
        frozenset(['R','G','W','U']): 'Altruism (RGWU)',
        frozenset(['G','W','U','B']): 'Growth (GWUB)',
    }
    key = frozenset(colors)
    if key in mapping:
        return mapping[key]
    if len(key) >= 5:
        return '5-Color (WUBRG)'
    return 'Other'

## Step 1.3: Load the deck .json files

In [13]:
CARD_SECTIONS = ['commander', 'mainBoard']
records = []

for filepath in glob.glob(f'{SLIM_DECKS_PATH}*.json'):
    with open(filepath, 'r', encoding=ENCODING) as f:
        raw = json.load(f)

    deck = raw.get('data', raw)

    #Combine card sets from selected card containing sections
    all_cards = [card for section in CARD_SECTIONS for card in deck.get(section, [])]
    card_names = [c['name'] for c in all_cards if 'name' in c]
    card_uuids = [c['uuid'] for c in all_cards if 'uuid' in c]

    #Determine colorIdentity for deck
    all_colors = set()
    for uuid in card_uuids:
        info = card_lookup.get(uuid)
        if info:
            all_colors.update(info['colorIdentity'])

    records.append({
        'deck_name': deck.get('name', os.path.basename(filepath)),
        'deck_type':      deck.get('type', 'Unknown'),
        'releaseDate':    deck.get('releaseDate', None),
        'card_names':     card_names,
        'deck_size':      len(card_names),
        'color_identity': color_label(all_colors),
    })

decks_df = pandas.DataFrame(records)
decks_df['releaseDate'] = pandas.to_datetime(decks_df['releaseDate'], errors='coerce')
decks_df['releaseYear'] = decks_df['releaseDate'].dt.year

print(f'Records loaded: {len(records)}')
print(f'Date range:    {decks_df["releaseDate"].min().date()} to {decks_df["releaseDate"].max().date()}')
decks_df[['deck_name','deck_type','releaseDate','deck_size','color_identity']].head()

Records loaded: 2654
Date range:    1993-12-10 to 2026-01-23


,deck_name,deck_type,releaseDate,deck_size,color_identity
0,2014 Full Art Land Set,Box Set,2014-01-01,5,Colorless
1,20 Ways to Win,Commander Deck,2024-12-04,100,Colorless
2,30th Anniversary Countdown Kit,Secret Lair Drop,2022-11-01,30,Colorless
3,Aang,Jumpstart,2025-11-20,15,Colorless
4,Above the Clouds (1),Jumpstart,2020-07-17,17,Colorless


## Step 2: EDA
### Step 2.1: Scope
First we lay out the scope of the data, so that we can make informed choices for the thresholds of the Aprioiri.